In [1]:
# -*- coding: utf-8 -*-
"""train_grpo_qwen_merged.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/github/standing-on-giants/Meta_hackathon/blob/dev%2Fpratham/train_grpo/train_grpo_qwen_merged.ipynb
"""

import os
import sys
import json
import textwrap
import random
import re
import subprocess
from pathlib import Path

# ==============================================================================
# 1. SETUP & INSTALLATION
# ==============================================================================
print("--- Starting Setup & Installation ---")

subprocess.run(
    "pip install -qU --no-cache-dir unsloth unsloth_zoo transformers trl peft accelerate "
    "bitsandbytes datasets pandas openai python-dotenv",
    shell=True, check=True
)

subprocess.run("rm -rf /kaggle/working/unsloth_compiled_cache", shell=True)
subprocess.run("rm -rf /kaggle/working/Meta_hackathon/unsloth_compiled_cache", shell=True)
print('Installation and Cache Clearance complete.')

# ✅ PATCH GOES HERE — after pip install, before anything else
import transformers.configuration_utils as _tcu
_orig_dumps = json.dumps
def _safe_dumps(obj, **kw):
    class SafeEncoder(json.JSONEncoder):
        def default(self, o):
            return None if callable(o) else super().default(o)
    kw['cls'] = SafeEncoder
    return _orig_dumps(obj, **kw)
_tcu.json.dumps = _safe_dumps
print("JSON encoder patched.")

try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    GITHUB_TOKEN = _s.get_secret('META_HACKATHON_TOKEN')
    HF_TOKEN = _s.get_secret('HF_TOKEN')   # ← matches your screenshot exactly
except Exception as e:
    print(f"Warning: Could not load secrets: {e}")
    GITHUB_TOKEN = os.getenv('META_HACKATHON_TOKEN', '')
    HF_TOKEN = os.getenv('HF_TOKEN', '')

# Normalize empty strings to None
HF_TOKEN = (HF_TOKEN or '').strip() or None
GITHUB_TOKEN = (GITHUB_TOKEN or '').strip() or None

print(f"HF_TOKEN loaded: {'YES' if HF_TOKEN else 'NO'}")
print(f"GITHUB_TOKEN loaded: {'YES' if GITHUB_TOKEN else 'NO'}")

REPO_DIR = '/kaggle/working/Meta_hackathon'

if not os.path.exists(REPO_DIR):
    subprocess.run(
        f"git clone -b dev/pratham https://{GITHUB_TOKEN}@github.com/"
        f"standing-on-giants/Meta_hackathon.git {REPO_DIR}",
        shell=True, check=True
    )
else:
    subprocess.run(
        "git fetch origin && "
        "git checkout -- src/__pycache__/ && "  # discard pycache conflicts
        "git checkout dev/pratham && "
        "git pull origin dev/pratham",
        shell=True, cwd=REPO_DIR
    )

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print(f'Working directory set to: {os.getcwd()}')

import torch
import numpy as np

from src.environment import DataPipelineEnv
from src.models import PipelineAction, PipelineObservation

print(f'PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}, VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB')

# ==============================================================================
# 2. LOADING THE MODEL (8-BIT QUANTIZATION)
# ==============================================================================
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 2048
MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'

print(f"--- Loading Base Model {MODEL_NAME} ---")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=False,
    load_in_8bit=True,
    token=HF_TOKEN,
)
print(f'Model loaded: {MODEL_NAME}')
print(f'Parameters: {model.num_parameters()/1e9:.2f}B')

model = FastLanguageModel.get_peft_model(
    model, r=32,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=32, lora_dropout=0.0, bias='none',
    use_gradient_checkpointing='unsloth', random_state=42,
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Trainable Parameters: {trainable/1e6:.1f}M / {total/1e9:.2f}B ({100*trainable/total:.2f}%)')

# === SMOKE TEST: Verify save works before training ===
import shutil
print("Running Smoke Test for Model Saves...")
try:
    model.save_pretrained('/kaggle/working/_smoke_test_lora')
    model.save_pretrained_merged('/kaggle/working/_smoke_test_merged', tokenizer, save_method='merged_16bit')

    # Verify config.json is valid
    with open('/kaggle/working/_smoke_test_merged/config.json', 'r') as f:
        json.loads(f.read())

    print("Smoke test PASSED — safe to proceed.")
    shutil.rmtree('/kaggle/working/_smoke_test_lora', ignore_errors=True)
    shutil.rmtree('/kaggle/working/_smoke_test_merged', ignore_errors=True)
except Exception as e:
    import traceback
    traceback.print_exc()
    raise RuntimeError(f"Smoke Test FAILED: {e}")

# ==============================================================================
# 3. SUPERVISED FINE-TUNING (SFT) - STAGE 1
# ==============================================================================
SYSTEM_PROMPT = textwrap.dedent('''
You are an expert data engineer diagnosing and fixing broken data pipelines.
You receive pipeline state and must choose ONE action per turn.
WORKFLOW: 1. read_data_sample 2. check_schema/compare_schema 3. Apply fix 4. run_pipeline
RULES: Respond with ONLY a JSON object. Never repeat failing actions. dedup for uniqueness failures.
''').strip()

# Shashank's Gold Actions map
GOLD_ACTIONS = {
    'easy': [
        {'action_type': 'read_data_sample', 'params': {'table': 'raw_orders', 'n_rows': 20}},
        {'action_type': 'add_data_filter', 'params': {'step_id': 'transform_orders', 'filter_condition': 'user_id IS NOT NULL'}},
        {'action_type': 'run_pipeline', 'params': {}},
    ],
    'medium': [
        {'action_type': 'read_data_sample', 'params': {'table': 'raw_order_items', 'n_rows': 20}},
        {'action_type': 'patch_transformation', 'params': {'step_id': 'transform_items', 'patch_type': 'dedup', 'column': 'order_item_id'}},
        {'action_type': 'run_pipeline', 'params': {}},
    ],
    'hard': [
        {'action_type': 'read_data_sample', 'params': {'table': 'raw_ads_insights', 'n_rows': 20}},
        {'action_type': 'compare_schema', 'params': {'table': 'raw_ads_insights'}},
        {'action_type': 'patch_transformation', 'params': {'step_id': 'transform_insights', 'patch_type': 'parse_currency', 'column': 'spend'}},
        {'action_type': 'patch_transformation', 'params': {'step_id': 'transform_insights', 'patch_type': 'coalesce', 'column': 'spend'}},
        {'action_type': 'read_data_sample', 'params': {'table': 'raw_conversions', 'n_rows': 20}},
        {'action_type': 'patch_transformation', 'params': {'step_id': 'transform_conversions', 'patch_type': 'dedup', 'column': 'event_id'}},
        {'action_type': 'compare_schema', 'params': {'table': 'raw_ads_insights'}},
        {'action_type': 'patch_transformation', 'params': {'step_id': 'transform_insights', 'patch_type': 'strip_prefix', 'column': 'campaign_id'}},
        {'action_type': 'alert_upstream_team', 'params': {'team': 'meta_ads_api_team', 'issue_description': 'N/A values in impressions cannot be parsed'}},
        {'action_type': 'run_pipeline', 'params': {}},
    ],
    'hard2': [
        {'action_type': 'read_data_sample', 'params': {'table': 'raw_ads_insights', 'n_rows': 20}},
        {'action_type': 'compare_schema', 'params': {'table': 'raw_ads_insights'}},
        {'action_type': 'patch_transformation', 'params': {'step_id': 'transform_insights', 'patch_type': 'parse_currency', 'column': 'spend'}},
        {'action_type': 'patch_transformation', 'params': {'step_id': 'transform_insights', 'patch_type': 'coalesce', 'column': 'spend'}},
        {'action_type': 'patch_transformation', 'params': {'step_id': 'transform_conversions', 'patch_type': 'dedup', 'column': 'event_id'}},
        {'action_type': 'run_pipeline', 'params': {}},
        {'action_type': 'compare_schema', 'params': {'table': 'raw_ads_insights'}},
        {'action_type': 'handle_drift', 'params': {'strategy': 'resolve_column_rename', 'table': 'raw_ads_insights', 'old_column': 'spend', 'new_column': 'total_spend'}},
        {'action_type': 'run_pipeline', 'params': {}},
    ],
}

def format_obs(obs, step):
    failed = '\n'.join(f'  [{r.assertion_id}] {r.assertion_type} on {r.table}({r.column or "N/A"}): {r.actual}' for r in obs.failed_assertions) or '  (none)'
    passed = ', '.join(r.assertion_id for r in obs.passed_assertions) or 'none'
    dag = '\n'.join(f'  {n.step_id}: {n.input_table} -> {n.output_table}' for n in obs.dag_structure)
    hist = '\n'.join(f'  {r.date}: {r.status} ({r.row_count} rows)' for r in obs.historical_runs)
    schema = ''
    if obs.current_schema: schema += '\nSCHEMA: ' + json.dumps(obs.current_schema)
    if obs.schema_diff: schema += '\nDIFF: ' + json.dumps(obs.schema_diff)
    sample = ''
    if obs.data_sample: sample = '\nDATA: ' + json.dumps(obs.data_sample[:3], default=str)
    actions = '\n'.join(f'  {a}' for a in obs.actions_taken[-4:]) or '  (none)'
    return f'STEP {step}/{obs.max_steps} | TASK: {obs.task_id} ({obs.difficulty})\nDESCRIPTION: {obs.description}\nPIPELINE PASSED: {obs.pipeline_passed}\nLAST RESULT: {obs.last_action_result}\nDAG:\n{dag}\nFAILING:\n{failed}\nPASSING: {passed}\nHISTORY:\n{hist}\nACTIONS:\n{actions}{sample}{schema}\nRespond with ONE action JSON.'

def collect_gold(task_ids=['easy','medium', 'hard', 'hard2'], n_ep=10):
    pairs = []
    for tid in task_ids:
        gold = GOLD_ACTIONS.get(tid, [])
        if not gold: continue
        for _ in range(n_ep):
            env = DataPipelineEnv(task_id=tid)
            res = env.reset()
            obs = res[0] if isinstance(res, tuple) else res
            for si, ad in enumerate(gold, 1):
                pairs.append((format_obs(obs, si), json.dumps(ad)))
                result = env.step(PipelineAction(**ad))
                obs = result.observation
                if obs.pipeline_passed: break
    return pairs

gold_pairs = collect_gold(n_ep=10)
print(f'Collected {len(gold_pairs)} SFT pairs')

from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

sft_texts = [
    tokenizer.apply_chat_template(
        [{'role':'system','content':SYSTEM_PROMPT},
         {'role':'user','content':obs},
         {'role':'assistant','content':act}],
        tokenize=False, add_generation_prompt=False)
    for obs, act in gold_pairs
]
sft_ds = Dataset.from_dict({'text': sft_texts})
sft_ds_split = sft_ds.train_test_split(test_size=0.1, seed=42)

SFT_DIR = '/kaggle/working/sft_qwen'

# Extreme VRAM preservation applied to SFT
print('--- Starting SFT ---')
# from transformers import EarlyStoppingCallback

sft_trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=sft_ds_split['train'],
    eval_dataset=sft_ds_split['test'],
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH,
    # callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    args=TrainingArguments(
        average_tokens_across_devices=False,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        num_train_epochs=2,
        eval_strategy='steps',
        eval_steps=20,
        # load_best_model_at_end=True,
        # metric_for_best_model='eval_loss',
        warmup_ratio=0.1,
        learning_rate=1e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=5, optim='adamw_8bit',
        weight_decay=0.05, lr_scheduler_type='cosine',
        output_dir=SFT_DIR, save_steps=40, seed=42,
    ),
)
sft_stats = sft_trainer.train()
print(f'SFT done. Loss: {sft_stats.training_loss:.4f}')
model.save_pretrained(SFT_DIR)
tokenizer.save_pretrained(SFT_DIR)

HF_REPO = 'Abhinav-hf/qwen-grpo-sft-trained-16bit'

if HF_TOKEN:
    print(f'Pushing seamlessly compiled SFT 16-bit model to Hub: {HF_REPO}')
    model.push_to_hub_merged(HF_REPO, tokenizer, save_method='merged_16bit', token=HF_TOKEN)
    print(f'SFT Hub upload complete. GRPO will now commence and overwrite the main weights once finished.')
else:
    print('No HF_TOKEN detected — skipping SFT Hub upload.')

# ==============================================================================
# 4. REINFORCEMENT LEARNING ON ENVIRONMENT (GRPO) - STAGE 2
# ==============================================================================
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
from src.models import PipelineAction, PipelineObservation

def parse_action(text):
    text = re.sub(r'<think>[\s\S]*?</think>', '', text, flags=re.DOTALL).strip()
    if '```' in text:
        text = '\n'.join(l for l in text.split('\n') if not l.strip().startswith('```')).strip()
    start = text.find('{')
    if start == -1: return None
    end = text.rfind('}') + 1
    if end <= start: return None
    try:
        data = json.loads(text[start:end])
        if 'action_type' in data: return PipelineAction(**data)
    except: pass
    return None

def run_episode(model, tokenizer, env, system_prompt, 
                max_steps=10, max_new_tokens=100):
    """
    Runs one full episode. Returns:
      - trajectory: list of (prompt_text, completion_text, step_reward)
      - total_reward: float
      - solved: bool
    """
    res = env.reset()
    obs = res[0] if isinstance(res, tuple) else res
    
    history = [{'role': 'system', 'content': system_prompt}]
    
    trajectory = []
    total_reward = 0.0
    solved = False
    
    for step in range(1, max_steps + 1):
        obs_text = format_obs(obs, step)
        history.append({'role': 'user', 'content': obs_text})
        
        prompt_text = tokenizer.apply_chat_template(
            history, tokenize=False, add_generation_prompt=True
        )
        
        inputs = tokenizer(
            prompt_text,
            return_tensors="pt",
            truncation=True,
            max_length=1800,
            add_special_tokens=True,
        ).to(model.device)
        
        with torch.no_grad():
            output_ids = model.generate(
                input_ids=inputs['input_ids'],
                attention_mask=inputs['attention_mask'],
                max_new_tokens=max_new_tokens,
                temperature=0.8,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
                use_cache=True,
            )
            
        new_tokens = output_ids[0][inputs.input_ids.shape[1]:]
        completion_text = tokenizer.decode(new_tokens, skip_special_tokens=True)
        
        action = parse_action(completion_text)
        
        if action is None:
            step_reward = -0.5
            trajectory.append((prompt_text, completion_text, step_reward))
            total_reward += step_reward
            break
            
        n_passed_before = len(obs.passed_assertions)
        result = env.step(action)
        obs = result.observation
        env_reward = result.reward or 0.0
        n_passed_after = len(obs.passed_assertions)
        
        step_reward = env_reward
        if n_passed_after > n_passed_before:
            step_reward += 0.15 * (n_passed_after - n_passed_before)
        elif step_reward <= 0:
            step_reward -= 0.1  # no change and no progress
        
        step_reward -= 0.02  # Reduced efficiency penalty
        
        trajectory.append((prompt_text, completion_text, step_reward))
        total_reward += step_reward
        
        history.append({'role': 'assistant', 'content': completion_text})
        
        if obs.pipeline_passed:
            trajectory[-1] = (prompt_text, completion_text, step_reward + 1.0)
            total_reward += 1.0
            solved = True
            break
            
    return trajectory, total_reward, solved

grpo_task_ids = ['easy', 'medium', 'hard', 'hard2']
N_EPISODES_PER_TASK = 60   # 4 tasks × 30 = 120 episodes per epoch
N_EPOCHS = 2
GRPO_LR = 1e-5
MAX_EPISODE_STEPS = 6

from bitsandbytes.optim import PagedAdamW8bit
optimizer = PagedAdamW8bit(
    [p for p in model.parameters() if p.requires_grad],
    lr=GRPO_LR, weight_decay=0.05
)

from transformers import get_cosine_schedule_with_warmup
total_steps = N_EPOCHS * N_EPISODES_PER_TASK * len(grpo_task_ids)
scheduler = get_cosine_schedule_with_warmup(
    optimizer, 
    num_warmup_steps=20, 
    num_training_steps=total_steps
)

training_log = []
GRPO_DIR = '/kaggle/working/grpo_qwen'

for epoch in range(N_EPOCHS):
    early_stop = False
    task_list = grpo_task_ids * N_EPISODES_PER_TASK
    random.shuffle(task_list)
    
    for step_idx, task_id in enumerate(task_list):
        model.eval()
        env = DataPipelineEnv(task_id=task_id, max_steps=MAX_EPISODE_STEPS)
        trajectory, total_reward, solved = run_episode(
            model, tokenizer, env, SYSTEM_PROMPT,
            max_steps=MAX_EPISODE_STEPS
        )
        
        if len(trajectory) == 0:
            continue
        
        torch.cuda.empty_cache()
        model.train()
        total_loss = torch.tensor(0.0, dtype=torch.float32, device=model.device)
        
        for prompt_text, completion_text, step_reward in trajectory:
            # Tokenize prompt and completion separately
            prompt_ids = tokenizer(
                prompt_text,
                return_tensors='pt',
                truncation=True,
                max_length=MAX_SEQ_LENGTH,
                add_special_tokens=False,
            ).input_ids.to(model.device)
            
            completion_ids = tokenizer(
                completion_text,
                return_tensors='pt',
                truncation=True,
                max_length=256,
                add_special_tokens=False,
            ).input_ids.to(model.device)
            
            # Keep only last 200 tokens of prompt (most recent state)
            MAX_PROMPT_TOKENS = 200
            prompt_ids = prompt_ids[:, -MAX_PROMPT_TOKENS:]
            
            # Concatenate: [recent_state | action]
            combined_ids = torch.cat([prompt_ids, completion_ids], dim=1)
            attention_mask = torch.ones_like(combined_ids)
            
            # Labels: mask prompt tokens (-100 = ignore in loss),
            # only train on completion tokens
            labels = combined_ids.clone()
            labels[:, :prompt_ids.shape[1]] = -100
            
            with torch.amp.autocast('cuda', dtype=torch.float16):
                outputs = model(
                    input_ids=combined_ids,
                    attention_mask=attention_mask,
                    labels=labels,
                )
            
            # Skip if loss is NaN (can happen with very short completions)
            if torch.isnan(outputs.loss):
                continue
                
            reward_tensor = torch.tensor(
                step_reward, dtype=torch.float32, device=model.device
            )
            step_loss = -reward_tensor * outputs.loss
            total_loss = total_loss + step_loss
        
        total_loss = total_loss / max(len(trajectory), 1)
        
        optimizer.zero_grad()
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(
            model.parameters(), max_norm=0.1
        )
        optimizer.step()
        scheduler.step()
        
        log_entry = {
            'step': step_idx + 1,
            'task': task_id,
            'total_reward': round(total_reward, 3),
            'solved': solved,
            'n_actions': len(trajectory),
            'loss': round(total_loss.item(), 4),
            'lr': scheduler.get_last_lr()[0]
        }
        training_log.append(log_entry)
        
        recent_window = training_log[-10:] if len(training_log) >= 10 \
                        else training_log
        avg_r = sum(x['total_reward'] for x in recent_window) / \
                len(recent_window)
        solved_pct = sum(x['solved'] for x in recent_window) / \
                     len(recent_window) * 100
        print(f"Ep {step_idx+1}/{total_steps} | task={task_id:<6} | "
              f"reward={log_entry['total_reward']:+.3f} | "
              f"avg10={avg_r:+.3f} | "
              f"solved={solved_pct:.0f}% | "
              f"steps={log_entry['n_actions']} | "
              f"loss={log_entry['loss']:.4f} | "
              f"lr={log_entry['lr']:.2e}")

        # Early stopping: if rolling solve rate > 60% over last 30 
        # episodes, training has converged
        if len(training_log) >= 30:
            last_30 = training_log[-30:]
            rolling_solve = sum(x['solved'] for x in last_30) / 30
            if rolling_solve >= 0.6:
                print(f"\\n✓ Early stopping triggered at episode "
                      f"{step_idx+1}: {rolling_solve*100:.0f}% "
                      f"solve rate over last 30 episodes.")
                early_stop = True
                model.save_pretrained(GRPO_DIR)
                break
        
        if (step_idx + 1) % 50 == 0:
            model.save_pretrained(GRPO_DIR)
            print(f"Checkpoint saved at step {step_idx+1}")

    if early_stop:
        break

model.save_pretrained(GRPO_DIR)
tokenizer.save_pretrained(GRPO_DIR)

total_solved = sum(x['solved'] for x in training_log)
print(f"\\nGRPO Complete.")
print(f"Episodes solved: {total_solved}/{len(training_log)} "
      f"({100*total_solved/len(training_log):.1f}%)")
print(f"Final avg reward (last 20): "
      f"{sum(x['total_reward'] for x in training_log[-20:])/20:.3f}")


# ==============================================================================
# 5. EXPORT / PUSH TO HUGGINGFACE HUB
# ==============================================================================

LOCAL_MERGED_DIR = '/kaggle/working/qwen-merged-16bit'
print(f'Saving completely merged 16-bit GRPO model locally to {LOCAL_MERGED_DIR}...')

model.save_pretrained_merged(
    LOCAL_MERGED_DIR,
    tokenizer,
    save_method='merged_16bit'
)
HF_REPO = 'Abhinav-hf/qwen-grpo-complete-trained-16bit'
if HF_TOKEN:
    print(f'Pushing seamlessly compiled GRPO model to Hub: {HF_REPO}')
    model.push_to_hub_merged(HF_REPO, tokenizer, save_method='merged_16bit', token=HF_TOKEN)
    print(f'Done! Final GRPO Model available at: https://huggingface.co/{HF_REPO}')
else:
    print('No HF_TOKEN detected — skipping Hub upload. Local save complete.')

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

if training_log:
    steps       = [x['step']         for x in training_log]
    rewards     = [x['total_reward'] for x in training_log]
    losses      = [x['loss']         for x in training_log]
    solved_flag = [1 if x['solved'] else 0 
                   for x in training_log]

    def rolling_avg(lst, w=10):
        return [
            sum(lst[max(0,i-w):i+1]) / 
            len(lst[max(0,i-w):i+1])
            for i in range(len(lst))
        ]

    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 4))

    ax1.plot(steps, rewards, alpha=0.3, color='green', 
             label='raw')
    ax1.plot(steps, rolling_avg(rewards), color='green',
             linewidth=2, label='smoothed')
    ax1.set_title('Episode Total Reward')
    ax1.set_xlabel('Episode')
    ax1.set_ylabel('Reward')
    ax1.legend()
    ax1.grid(True)

    ax2.plot(steps, losses, alpha=0.4, color='blue')
    ax2.set_title('Training Loss per Episode')
    ax2.set_xlabel('Episode')
    ax2.set_ylabel('Loss')
    ax2.grid(True)

    ax3.plot(steps, rolling_avg(solved_flag, w=20), 
             color='orange', linewidth=2)
    ax3.set_title('Solve Rate (rolling 20)')
    ax3.set_xlabel('Episode')
    ax3.set_ylabel('Fraction Solved')
    ax3.set_ylim(0, 1)
    ax3.grid(True)

    plt.tight_layout()
    plt.savefig('/kaggle/working/grpo_training_curves.png', 
                dpi=150)
    print('Saved: /kaggle/working/grpo_training_curves.png')
else:
    print('No training log to plot.')

--- Starting Setup & Installation ---
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 84.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 303.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 179.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 376.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 205.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 331.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 170.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 352.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 209.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 354.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 279.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 318.4 MB/s eta 0:00:

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
s3fs 2026.2.0 requires fsspec==2026.2.0, but you have fsspec 2025.9.0 which is incompatible.
ydata-profiling 4.18.1 requires pandas!=1.4.0,<3.0,>1.5, but you have pandas 3.0.2 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
db-dtypes 1.5.0 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.2 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you ha

Installation and Cache Clearance complete.
JSON encoder patched.
HF_TOKEN loaded: YES
GITHUB_TOKEN loaded: YES


Cloning into '/kaggle/working/Meta_hackathon'...


Working directory set to: /kaggle/working/Meta_hackathon
PyTorch 2.10.0+cu128, CUDA: True
GPU: Tesla T4, VRAM: 15.6GB
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


...


Unsloth: Merging weights into 16bit:  50%|█████     | 1/2 [00:54<00:54, 54.40s/it]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [01:21<00:00, 40.74s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/Meta_hackathon/Abhinav-hf/qwen-grpo-complete-trained-16bit`
Done! Final GRPO Model available at: https://huggingface.co/Abhinav-hf/qwen-grpo-complete-trained-16bit
Saved: /kaggle/working/grpo_training_curves.png
